# EvalRX on a Colab TPU

Run Gemma 4 E2B with `jax_local`, generate answers, inspect attention and hidden states, and run a logit lens analyzer. No judge API key or SSH tunnel is needed for these checks.

Select **Runtime → Change runtime type → TPU**, then start with a fresh runtime. Use Python 3.12 or newer. Start with the text suite on a single 16 GB TPU; image and audio checks are optional.

Model calls run in separate Python processes so each finished suite releases its TPU buffers and compiled programs. Keep JAX imports out of the notebook kernel when using this workflow. This tutorial checks the backend and one analyzer, not the agent-driven M1–M5 repair loop.

## Get the source

The default ref is the `ruinan` development branch. For repeatable runs, replace `REF` with a commit that contains this notebook. An existing checkout is used as-is; its files are not reset. The checkout must include `tools/colab/setup.sh` and `tools/colab/jax_smoke.py`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REF = "ruinan"
repo = Path(os.environ.get("EVALRX_REPO_DIR", "/content/evalrx"))
if not repo.exists():
    subprocess.run(["git", "clone", "https://github.com/evalvitals/evalrx.git", str(repo)], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", REF], check=True)
assert (repo / "tools/colab/setup.sh").is_file(), "Use the tutorial's source branch or commit"
print("Python:", sys.version)
if (repo / ".git").exists():
    subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)
print("Source:", repo)

## Install the TPU dependencies

The installer preserves Colab's matched JAX, jaxlib and libtpu versions, its CPU PyTorch, and its fsspec version. It installs Gemma 4.0.1 and EvalRX from this checkout. Installation and the first model load can take several minutes. Logs are saved under `~/.cache/evalrx/colab`.

If you already imported Gemma or TensorFlow before installation, restart the Python session and rerun the cells. Do not upgrade `jax` or install CUDA wheels in this TPU runtime.

In [ ]:
subprocess.run(["bash", str(repo / "tools/colab/setup.sh")], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

## Run the text checks

This loads the public Gemma E2B checkpoint directly from Google Cloud Storage. It checks greedy and sampled generation, token log probabilities, finite logits and hidden states, normalized attention, and a logit lens analyzer. A nonzero exit or any failed step means the suite did not pass.

The first generation includes XLA compilation; the second reuses it. `forward_capture` returns CPU PyTorch tensors for EvalRX analyzers while the model runs on the TPU.

In [ ]:
import json

reports = repo / "colab_results"
reports.mkdir(exist_ok=True)
env = os.environ.copy()
env.update(JAX_PLATFORMS="tpu,cpu", OMP_NUM_THREADS="2", TF_CPP_MIN_LOG_LEVEL="2")

def run_suite(suite):
    output = reports / f"{suite}.json"
    command = [sys.executable, "-u", str(repo / "tools/colab/jax_smoke.py"),
               "--device", "tpu", "--suite", suite, "--output", str(output)]
    if suite == "text":
        command.append("--text-only")
    subprocess.run(command, cwd=repo, env=env, check=True)
    report = json.loads(output.read_text())
    assert report["status"] == "passed"
    for step in report["steps"]:
        print(f"{step['name']}: {step['status']} ({step['seconds']:.1f} s)")
    return report

text_report = run_suite("text")

## Read the results

Each JSON report includes the actual device, package versions, per-step duration, generated text, tensor shapes and memory counters. `running` means the process has not completed, even when earlier steps passed. The small prompts are functional checks, not an accuracy benchmark.

In [ ]:
steps = {s["name"]: s for s in text_report["steps"]}
print("Device:", steps["hardware"]["detail"]["devices"])
print("Answer:", steps["generate_greedy_warm"]["detail"]["answer"])
print("Capture:", json.dumps(steps["forward_capture"]["detail"], indent=2))
print("Logit lens:", json.dumps(steps["logit_lens_analyzer"]["detail"], indent=2))
print("Reports:", reports)

## Optional image and audio checks

These load the vision and audio towers and use more memory. A solid red image checks image encoding and generation; two seconds of silence check audio encoding and generation. They are plumbing checks, not evidence of real-world multimodal accuracy. Run them sequentially to free the TPU between suites.

The validated audio run peaked at **46.6 GB host RAM**, despite using only about 11.3 GB of TPU memory. Keep audio optional on runtimes with less host memory. Measured versions and results are in the repository's `docs/colab_tpu.md`.

In [ ]:
RUN_MULTIMODAL = False  # Set True to run both additional suites.
if RUN_MULTIMODAL:
    image_report = run_suite("image")
    audio_report = run_suite("audio")

## Use the Python API

The same API works in your own script. This optional cell executes a custom prompt in a separate process. `text_only=True` skips the media towers; use `False` with `Inputs(image=..., prompt=...)` or `Inputs(audio=..., prompt=...)` for multimodal inputs. Model inference uses JAX; captured arrays and the logit lens use CPU PyTorch.

In [ ]:
RUN_CUSTOM_PROMPT = False
if RUN_CUSTOM_PROMPT:
    demo = """
from evalrx.models import RuntimeConfig, compose
from evalrx.core.capability import Capability
from evalrx.analyzers.lens.logit_lens import LogitLensAnalyzer

model = compose("gemma-4-e2b-it", "jax_local", RuntimeConfig(
    device="tpu", dtype="bfloat16", apply_chat_template=True,
    max_new_tokens=32, engine_kwargs={"text_only": True}))
prompt = "What is the capital of France? Answer in one word."
print(model.generate(prompt))
trace = model.forward(prompt, capture={Capability.LOGITS, Capability.HIDDEN_STATES})
print(trace.logits.shape, len(trace.hidden_states), trace.logits.device)
print(LogitLensAnalyzer(top_k=3).run(model, prompt).to_json(indent=2))
"""
    subprocess.run([sys.executable, "-u", "-c", demo], cwd=repo, env=env, check=True)

## Save the reports

Download these before the Colab runtime expires. JAX model internals are supported for Gemma E2B/E4B, but this tutorial validates E2B on one device. Multi-chip FSDP, E4B, gradient analyzers, JAX LoRA and the full agent repair loop are not validated here. The JAX benchmark repair ladder currently stops at L2.

In [ ]:
import zipfile

archive = reports / "evalrx_tpu_reports.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(reports.glob("*.json")):
        z.write(path, arcname=path.name)
print("Saved", archive)
# In Colab, uncomment to download:
# from google.colab import files
# files.download(str(archive))